# 5.6 Your turn: does Louis write shorter messages in French?

Draft for the graded visualisation about **relationships** (lesson 5).

> **Question:** in weeks where Louis writes more in his mother tongue, French, are his messages shorter?

One dot per calendar week, Louis's messages only:

- **x** = share of that week's messages in French (among the messages long enough to classify, the same rule as figures 1 and 2 of the pipeline: under 3 characters is "unknown");
- **y** = words per message that week (all of his messages, the short ones included).

Two views: the whole chat, and only the long-distance time (before the test, and pregnant before the move).

Data: `2_features.parquet` from stage 2 of [roos_pipeline](../../roos_pipeline/README.md), which already holds `language` per message. Dates and colours come from `roos_pipeline/settings.py`.

In [ ]:
import numpy as np
import pandas as pd
from goad_toolkit.datatransforms import Filter, MapValues, Pipeline, RegexFeature
from goad_toolkit.visualizer import PlotSettings, RegPlot, ScatterPlot
from matplotlib.ticker import MultipleLocator, PercentFormatter
from scipy import stats

from roos_pipeline import settings
from roos_pipeline.settings import Paths

features = pd.read_parquet(Paths().features)
features.shape

## 5.6.1 Per message: Louis, word count, French or not

Three goad steps, used as-is:

- `Filter` keeps Louis's messages.
- `RegexFeature` counts words: `\S+` is one run of characters without a space, so an emoji or a link counts as one word.
- `MapValues` turns `language` into `is_french`: 1.0 for French, 0.0 for English or Dutch, and NaN for "unknown". A NaN is skipped by the weekly mean, so the share is over classified messages only.

In [ ]:
louis = (
    Pipeline()
    .add(Filter, expr="author == 'Louis'")
    .add(RegexFeature, name="words", column="message", pattern=r"\S+",
         feature="n_words", mode="count")
    .add(MapValues, column="language", mapping={"fr": 1.0, "en": 0.0, "nl": 0.0},
         feature="is_french")
    .apply(features)
)
# The week each message belongs to, labelled by its Sunday: the same label resample("W") gives.
louis["week"] = louis["ts_local"].dt.to_period("W-SUN").dt.end_time.dt.normalize()
louis[["ts_local", "message", "language", "n_words", "is_french"]].sample(8, random_state=settings.SEED)

## 5.6.2 Per week

The grain changes here: one row per week instead of one per message. Same week convention as `weekly_topics` in the pipeline (`resample("W")`, each week labelled by its Sunday, so it starts six days earlier).

Each week gets two labels:

- `living`: **long distance** (the week ends before the move) or **living together** (it starts after). The one week that holds the move is half of each, so it is neither.
- `period`: long distance splits once more at the positive test, for the counts in the second plot. The week of the test is still long distance.

Weeks with very few messages give noisy dots: one long message makes a week of 3 messages look very long. `MIN_MESSAGES` leaves those weeks out. **10 is a first guess, not a decision yet.**

In [ ]:
MIN_MESSAGES = 10

LONG_DISTANCE = "long distance"
TOGETHER = "living together"
MOVE_WEEK = "week of the move"
PREGNANT = "long distance, pregnant"

by_week = louis.set_index("ts_local").resample("W")
weekly = pd.DataFrame({
    "french_share": by_week["is_french"].mean(),
    "words_per_message": by_week["n_words"].mean(),
    "median_words": by_week["n_words"].median(),
    "n_messages": by_week.size(),
}).rename_axis("week").reset_index()

week_end = weekly["week"]
week_start = week_end - pd.Timedelta(days=6)
weekly["living"] = np.select(
    [week_end < settings.MOVE_DATE, week_start >= settings.MOVE_DATE],
    [LONG_DISTANCE, TOGETHER],
    default=MOVE_WEEK,
)
weekly["period"] = np.where(
    (weekly["living"] == LONG_DISTANCE) & (week_start >= settings.TEST_DATE),
    PREGNANT,
    weekly["living"],
)

enough = weekly[weekly["n_messages"] >= MIN_MESSAGES].dropna(subset=["french_share"])
print(f"{len(weekly)} weeks, {len(enough)} with at least {MIN_MESSAGES} messages from Louis")
enough.groupby("period").agg(
    weeks=("week", "size"),
    french_share=("french_share", "mean"),
    words_per_message=("words_per_message", "mean"),
).round(2)

The caption under each graph: how many of Louis's messages the dots rest on, and which share was English, Dutch and French. Percentages are over the classified messages, so they add up to 100%; the "unknown" ones are counted separately.

In [ ]:
LANGUAGE_NAMES = {"en": "English", "nl": "Dutch", "fr": "French"}


def language_caption(messages: pd.DataFrame, weeks: int) -> str:
    counts = messages["language"].value_counts()
    known = counts.drop("unknown", errors="ignore")
    pct = 100 * known / known.sum()
    shares = ", ".join(f"{pct.get(code, 0):.0f}% {name}" for code, name in LANGUAGE_NAMES.items())
    return (
        f"Based on {len(messages):,} messages from Louis over {weeks} weeks: {shares}\n"
        f"(of the {known.sum():,} messages long enough to classify; "
        f"{counts.get('unknown', 0):,} messages under {settings.MIN_CHARS} characters "
        f"have no language but do count for words per message)."
    )


def messages_in(weeks: pd.DataFrame) -> pd.DataFrame:
    return louis[louis["week"].isin(weeks["week"])]

## 5.6.3 The whole period

Orange: the weeks we lived apart. Blue: the weeks we lived together. One line through all of them.

In [ ]:
x, y = "french_share", "words_per_message"
r_all = enough[x].corr(enough[y])
LIVING_COLOURS = {
    LONG_DISTANCE: settings.LONG_DISTANCE_ORANGE,
    TOGETHER: settings.TOGETHER_BLUE,
    MOVE_WEEK: settings.LIGHT_GREY,
}

whole = PlotSettings(
    figsize=(8, 5.5),
    title="Does Louis write shorter messages in French?",
    xlabel="share of messages in French that week",
    ylabel="words per message",
)
host = ScatterPlot(whole)
fig, ax = host.plot(
    data=enough, x=x, y=y, hue="living", palette=LIVING_COLOURS,
    hue_order=[g for g in LIVING_COLOURS if g in set(enough["living"])], alpha=0.7,
)
host.plot_on_axes(RegPlot(whole), ax, data=enough, x=x, y=y, scatter=False, ci=None,
                  color=settings.DARK_GREY)
ax.set(xlabel=whole.xlabel, ylabel=whole.ylabel)  # seaborn writes the column names over them
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.legend(title="", loc="upper center")
ax.text(0.99, 0.98, f"{len(enough)} weeks, whole chat\nr = {r_all:+.2f}",
        transform=ax.transAxes, ha="right", va="top", color=settings.GREY)
fig.text(0.5, -0.02, language_caption(messages_in(enough), len(enough)),
         ha="center", va="top", fontsize=9, color="dimgray");

## 5.6.4 Only the long-distance time

The orange dots of the first graph on their own: the weeks before the positive test and the pregnant weeks before the move (until 1 Nov 2024). The week of the move is left out.

`LINE_FIT` decides how the line is drawn. It is a straight line for now (`{}`): **pick one of the options in 5.6.5** and put it here, e.g. `{"logx": True}` or `{"lowess": True}`.

In [ ]:
LINE_FIT: dict = {}

apart = enough[enough["living"] == LONG_DISTANCE]
r_apart = apart[x].corr(apart[y])

long_distance = PlotSettings(
    figsize=(8, 5.5),
    title="Does Louis write shorter messages in French? Long distance only",
    xlabel="share of messages in French that week",
    ylabel="words per message",
)
fig, ax = RegPlot(long_distance).plot(
    data=apart, x=x, y=y, ci=None, color=settings.DARK_GREY,
    scatter_kws={"color": settings.LONG_DISTANCE_ORANGE, "alpha": 0.7}, **LINE_FIT,
)
ax.set(xlabel=long_distance.xlabel, ylabel=long_distance.ylabel)
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
n_pregnant = (apart["period"] == PREGNANT).sum()
ax.text(0.99, 0.80,
        f"{len(apart)} weeks long distance\n"
        f"of which {n_pregnant} pregnant\n"
        f"r = {r_apart:+.2f}",
        transform=ax.transAxes, ha="right", va="top", color=settings.GREY)
fig.text(0.5, -0.02, language_caption(messages_in(apart), len(apart)),
         ha="center", va="top", fontsize=9, color="dimgray");

### What is one dot?

**One dot is one calendar week** (Monday to Sunday) while we lived apart, and only **Louis's** messages in that week:

- **left–right (x):** of his messages that week that got a language, the share that was French. A dot at 10% means 1 in 10 of his classified messages that week was French.
- **up–down (y):** the *mean* words per message that week: all the words he wrote that week, divided by the number of messages he sent (short "unknown" messages included).

So a dot does not show one message, and it does not show the French messages on their own: it mixes his French, English and Dutch messages of that week. The line says *weeks with more French are weeks with shorter messages on average*. Whether his *French messages themselves* are shorter is a different question, the per-message test (1) and (2) below.

The week below is the one closest to the middle of the long-distance French shares, as an example:

In [ ]:
example = apart.iloc[(apart[x] - apart[x].median()).abs().argsort().iloc[0]]
week_msgs = louis[louis["week"] == example["week"]]
print(f"week {example['week'] - pd.Timedelta(days=6):%d %b} - {example['week']:%d %b %Y}")
print(f"  {len(week_msgs)} messages from Louis, of which {week_msgs['is_french'].notna().sum()} with a language")
print(f"  {int(week_msgs['is_french'].sum())} French  ->  x = {example[x]:.0%}")
print(f"  {week_msgs['n_words'].sum():,} words / {len(week_msgs)} messages  ->  y = {example[y]:.1f} (mean)")
print(f"  median message: {example['median_words']:.0f} words")

## 5.6.4b The same weeks, median instead of mean

Each dot is the same week as above, but y is now the **median** words per message: the middle message of the week when his messages are lined up from short to long. One very long message (a quoted reply, a long story) moves the mean but hardly moves the median.

Note: a word count is a whole number, so a weekly median can only be a whole number or a half. Expect dots on horizontal rows.

In [ ]:
y_median = "median_words"
r_apart_median = apart[x].corr(apart[y_median])

long_distance_median = PlotSettings(
    figsize=(8, 5.5),
    title="Does Louis write shorter messages in French? Long distance only, median",
    xlabel="share of messages in French that week",
    ylabel="median words per message",
)
fig, ax = RegPlot(long_distance_median).plot(
    data=apart, x=x, y=y_median, ci=None, color=settings.DARK_GREY,
    scatter_kws={"color": settings.LONG_DISTANCE_ORANGE, "alpha": 0.7}, **LINE_FIT,
)
ax.set(xlabel=long_distance_median.xlabel, ylabel=long_distance_median.ylabel)
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.text(0.99, 0.98,
        f"{len(apart)} weeks long distance\n"
        f"of which {n_pregnant} pregnant\n"
        f"r = {r_apart_median:+.2f}",
        transform=ax.transAxes, ha="right", va="top", color=settings.GREY)
fig.text(0.5, -0.02, language_caption(messages_in(apart), len(apart)),
         ha="center", va="top", fontsize=9, color="dimgray");

In [ ]:
pd.DataFrame({
    "r with French share": [r_apart, r_apart_median],
    "lowest week": [apart[y].min(), apart[y_median].min()],
    "highest week": [apart[y].max(), apart[y_median].max()],
    "middle week": [apart[y].median(), apart[y_median].median()],
}, index=["mean words per message", "median words per message"]).round(2)

## 5.6.5 Four ways to draw the line

Every line is a claim about the relationship, not decoration (goad's `RegPlot` docstring: *the choice of fit is a claim about the relationship, not a formatting option*). The same long-distance weeks four times:

| Option | `LINE_FIT` | What it claims | Watch out |
|---|---|---|---|
| **Straight** | `{}` | every extra 10% French costs the same number of words | goes on falling forever: at the slope of the second graph, a week of 50% French would have *no* words per message, while the first graph shows about 6 there |
| **Log of x** | `{"logx": True}` | words = a + b · log(French share): the *first* bit of French matters most, then it flattens | needs a French share above 0 (every long-distance week has one); still one number (b) you can report |
| **Parabola** | `{"order": 2}` | a curve that bends once | can bend back *up* at the edge, because there are only a few dots above 25% |
| **Lowess** | `{"lowess": True}` | no formula: a local average that lets the data choose the shape | good for *looking*, but there is no slope to report, and it follows every bump |

Also possible on any of them: `{"robust": True}` lowers the weight of outliers, such as the one week around 21% French with almost 20 words per message.

Questions to decide with: which shape do you expect *before* looking (a reason, not a picture)? And does it still make sense when you put it next to the first graph, where French goes up to 85%?

In [ ]:
LINE_OPTIONS = {
    "straight: words = a + b · share": {},
    "log of x: words = a + b · log(share)": {"logx": True},
    "parabola: order = 2": {"order": 2},
    "lowess: the data choose the shape": {"lowess": True},
}

four_lines = PlotSettings(
    figsize=(12, 9),  # ty: ignore[invalid-argument-type]
    title="The same long-distance weeks, four lines",
    subplot_titles=list(LINE_OPTIONS),
    xlabel="share of messages in French that week",
    ylabel="words per message",
    max_cols=2,
    sharex=True,
    sharey=True,
)
host = ScatterPlot(four_lines)
fig, axes = host.create_figure(n_plots=len(LINE_OPTIONS))
for ax, fit in zip(axes, LINE_OPTIONS.values()):
    host.plot_on_axes(RegPlot(four_lines), ax, data=apart, x=x, y=y, ci=None,
                      color=settings.DARK_GREY,
                      scatter_kws={"color": settings.LONG_DISTANCE_ORANGE, "alpha": 0.7}, **fit)
    panel = ax.get_subplotspec()  # labels only on the outer edge: the axes are shared
    ax.set(xlabel=four_lines.xlabel if panel.is_last_row() else "",
           ylabel=four_lines.ylabel if panel.is_first_col() else "")
    ax.xaxis.set_major_formatter(PercentFormatter(1.0))

## 5.6.6 Test 2: within each week, are his French messages shorter?

**Why this test.** The downward slope in the scatter can come from the *mix*: if his French messages are shorter than his English and Dutch ones, a week with more French has a lower average by arithmetic alone. Test 2 checks that mechanism directly. It also removes the biggest confounder by design: everything that is the same within one week (long distance or not, pregnant or not, how busy, what was going on) is the same for his French and his other messages of that week, so it cannot make the difference.

**How it works.**

1. Take one long-distance week. Split Louis's classified messages of that week in two: French, and English + Dutch.
2. Take the median word count of each group, and subtract: **French minus other**. Below 0 means his French messages were shorter *that week*. A week needs at least `MIN_PER_GROUP` messages in each group, otherwise one message would be "the median".
3. Repeat for every week: one difference per week.
4. One question: **are these differences below 0?** The Wilcoxon signed-rank test answers it: if language made no difference, a week would be equally likely to land above as below 0, and big differences would be as common on one side as on the other. It ranks the differences by size and asks whether the negative ones carry far more weight than chance would give them. One-sided (`alternative="less"`), because the hypothesis has a direction.
5. The sign test is the plainest version of the same idea: count the weeks below 0. If chance decided, each week is a coin flip.

**What I expected before running it:** below 0, i.e. French messages shorter within the same week.

In [ ]:
MIN_PER_GROUP = 5


def within_week_differences(messages: pd.DataFrame, min_per_group: int = MIN_PER_GROUP) -> pd.Series:
    # Per week: median words of his French messages minus median words of his other ones.
    classified = messages[messages["is_french"].notna()]
    group = np.where(classified["is_french"] == 1.0, "French", "other")
    per_week = classified.groupby(["week", group])["n_words"].agg(["median", "size"]).unstack()
    both = per_week[(per_week[("size", "French")] >= min_per_group)
                    & (per_week[("size", "other")] >= min_per_group)]
    return (both[("median", "French")] - both[("median", "other")]).rename("difference")


apart_msgs = messages_in(apart)
diff = within_week_differences(apart_msgs)
wilcoxon = stats.wilcoxon(diff, alternative="less")
n_below = int((diff < 0).sum())
sign_test = stats.binomtest(n_below, n=int((diff != 0).sum()), p=0.5, alternative="greater")

print(f"{len(diff)} of {len(apart)} long-distance weeks have at least {MIN_PER_GROUP} messages in both groups")
print(f"French minus other, median words per message: median difference {diff.median():+.1f} words")
print(f"weeks below 0: {n_below} of {len(diff)}   (equal: {(diff == 0).sum()}, above: {(diff > 0).sum()})")
print(f"Wilcoxon signed-rank, one-sided: p = {wilcoxon.pvalue:.1e}")
print(f"sign test (coin flip per week):  p = {sign_test.pvalue:.1e}")
diff.describe().round(1)

**Outcome.** In every long-distance week with enough messages in both groups, his French messages had a lower median word count than his English and Dutch messages of the same week, typically 4 words fewer. If language made no difference, that would be like 63 coin flips all landing on the same side. Both p-values are far below 0.001.

**What it does not say.**

- *Why.* "Familiar language" is one explanation. Others: what he writes in French (quick practical messages?), or when he switches (a reply to a French message of mine?). The test shows *shorter*, not *because familiar*.
- Neighbouring weeks are not fully independent, so the exact p-value is too optimistic. With every week on the same side, the conclusion does not hinge on it.

### Is it the language detector or the quoted replies?

Two ways the result could be an artefact:

- `py3langid` is least reliable on short messages, so a short English message wrongly called French would make "French" look short.
- Quoted replies carry someone else's words, and they are more often in English.

So: the same test without quoted replies, with only messages of at least 4 words, and both.

In [ ]:
quoted = apart_msgs["message"].str.contains(settings.QUOTED_REPLY_PATTERN, regex=True)
checks = {
    "all classified messages": apart_msgs,
    "without quoted replies": apart_msgs[~quoted],
    "at least 4 words": apart_msgs[apart_msgs["n_words"] >= 4],
    "both": apart_msgs[~quoted & (apart_msgs["n_words"] >= 4)],
}
rows = []
for label, msgs in checks.items():
    d = within_week_differences(msgs)
    rows.append({
        "messages": label,
        "weeks": len(d),
        "weeks below 0": int((d < 0).sum()),
        "median difference": d.median(),
        "Wilcoxon p": stats.wilcoxon(d, alternative="less").pvalue,
    })
pd.DataFrame(rows).set_index("messages").style.format({"median difference": "{:+.1f}", "Wilcoxon p": "{:.0e}"})

The difference holds in all four versions. Leaving out the shortest messages makes it smaller (about 2.5 words instead of 4), so part of the gap sits in the very short messages, where the detector is least sure. Still, most weeks stay below 0.

## 5.6.7 The final graph

**The line: `words = a + b · log(share)`.** The fitted line is straight in the *logarithm* of the French share, so it is a curve on this axis. What that means: every time the French share **doubles** (5% → 10%, 10% → 20%), the average message loses the same number of words. Going from 5% to 10% French costs as much as going from 10% to 20%. So the first bit of French matters most in percentage points, and the curve flattens towards the right, instead of a straight line that would reach 0 words around 35% French.

In [ ]:
TITLE = "Familiar language needs fewer words"
SUBTITLE = "While we lived apart, Louis wrote shorter messages when he wrote in French."

b, a = np.polyfit(np.log(apart[x]), apart[y], 1)
per_doubling = b * np.log(2)
print(f"words per message = {a:.1f} {b:+.2f} · ln(French share)")
print(f"every doubling of the French share: {per_doubling:+.1f} words per message")

final = PlotSettings(
    figsize=(9, 6),
    title="",
    xlabel="share of messages in French that week",
    ylabel="words per message (average that week)",
)
fig, ax = RegPlot(final).plot(
    data=apart, x=x, y=y, logx=True, ci=None, color=settings.DARK_GREY,
    scatter_kws={"color": settings.LONG_DISTANCE_ORANGE, "alpha": 0.75},
)
ax.set(xlabel=final.xlabel, ylabel=final.ylabel)
ax.xaxis.set_major_locator(MultipleLocator(0.05))
ax.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
ax.text(0.98, 0.80, f"line: every doubling of French\n= {abs(per_doubling):.1f} fewer words per message",
        transform=ax.transAxes, ha="right", va="top", color=settings.DARK_GREY)

fig.subplots_adjust(top=0.84)
fig.suptitle(TITLE, fontsize=18, fontweight="bold", y=0.98)
fig.text(0.5, 0.905, SUBTITLE, ha="center", fontsize=11)

first_day = apart["week"].min() - pd.Timedelta(days=6)
if not (wilcoxon.pvalue < 0.001 and diff.median() < 0):
    raise ValueError("The test no longer supports the line under the graph: rewrite it.")
footnote = "\n".join([
    f"Each dot is one week we lived apart ({first_day:%b %Y} – {apart['week'].max():%b %Y}): "
    "what share of Louis's messages that week was French, and how many words his messages had on average.",
    language_caption(apart_msgs, len(apart)),
    f"Within the same week, his French messages were shorter than his English and Dutch ones in "
    f"{n_below} of {len(diff)} weeks (typically {abs(diff.median()):.0f} words fewer; Wilcoxon signed-rank, p < 0.001).",
])
fig.text(0.5, -0.02, footnote, ha="center", va="top", fontsize=8.5, color="dimgray", linespacing=1.5);

## What I see

*(Roos: in your own words. What does each plot show, and do the two agree? Which line, and why?)*

## How could I test it? (options, not chosen yet)

The claim has two parts: **(1)** Louis writes little French while we live apart, and **(2)** his English and Dutch messages have more words than his French ones. A test cannot *prove* either; it can show that chance alone is an unlikely explanation, and that the result survives the other explanations.

1. **Per message, the direct test of part 2.** Words per message, French against English + Dutch. Message length has a long tail, so compare medians: a Mann–Whitney U test (`scipy.stats.mannwhitneyu`), or the shuffle you already know: `NullDistribution` with the language label shuffled. Weak spot: messages are not independent (one conversation, one mood, one topic), so the p-value comes out too small.
2. **Within each week (05.2's *within species*).** Per week: median words of his French messages minus median words of his other messages. Then test whether those differences are below 0: a Wilcoxon signed-rank test (`scipy.stats.wilcoxon`), or flip the sign of each week at random. Everything that is the same within a week (period, how busy, how far apart) drops out by itself.
3. **Per week, the scatter itself.** Is r really different from 0? A Spearman correlation (`scipy.stats.spearmanr`, no straight line needed), or shuffle the French shares over the weeks. Neighbouring weeks look alike, so add the time-order check from 04.6d (runs of consecutive weeks).
4. **Several factors at once.** A model of the word count per message with French *and* the other suspects in it: period, hour of the day, topic, whether the message quotes an earlier one. Word counts are counts with a long tail, so a negative binomial regression (`statsmodels`), the same family as figure 5. The coefficient for French then says what French does *with the rest held fixed*.
5. **Part 1** (little French while apart): the language share per period is a count table, so a chi-square test (`scipy.stats.chi2_contingency`) on language × long distance / together.

Before running any of them (tip 7 from lesson 4): write down what difference would count, e.g. "his French messages are at least 2 words shorter, median, within the same week".

## Choices I still have to be able to defend

- **Mean or median words per message?** Message length has a long tail (lesson 4): one long message pulls a week's mean up. 5.6.4b draws both.
- **`MIN_MESSAGES = 10`:** right now it drops no week at all (Louis wrote at least 10 messages every week). Keep it as a guard, or remove it?
- **Quoted replies:** 2,396 of Louis's messages contain a quoted earlier message (`[18-12-2025, 09:42:27] Roos: ...`), and those words are counted as his. Strip the quote first, or leave those messages out?
- **"unknown" left out of the French share, but kept in words per message.** Short messages ("Ok", one emoji) are exactly the short ones: is that fair?
- **Which line** (5.6.5), and why that shape.
- **What else is true of the French weeks?** (05.2's question.) After the move his French share went up *and* we lived together, so there was less to say by text. Is the line about French, or about the period? The colours in the first graph and the second graph are two ways to check.